# Treino do Modelo de Classificação de Língua Gestual (ASL)

## Contexto e Objetivos
Este notebook corresponde à **Fase 2** do Trabalho Prático. O objetivo é desenvolver, treinar e avaliar um modelo de Aprendizagem Automática capaz de classificar caracteres do alfabeto de Língua Gestual Americana (ASL) (A-Z) com base em coordenadas de *landmarks* das mãos.

## Metodologia
De acordo com os requisitos do projeto, o fluxo de trabalho segue as etapas:
1.  **Carregamento e Pré-processamento:** Leitura do dataset estruturado gerado na Fase 1 e normalização dos dados.
2.  **Seleção de Algoritmos:** Comparação de múltiplos modelos exigidos (Random Forest, SVM, KNN, Árvores de Decisão e Redes Neuronais) utilizando Validação Cruzada.
3.  **Otimização:** Afinação de hiperparâmetros (Grid Search) para os modelos mais promissores.
4.  **Critério de Seleção:** Avaliação baseada em Accuracy, F1-Score e Tempo de Inferência.
5.  **Persistência:** Guardar o modelo final, o scaler e o encoder para integração na API Flask (Fase 3).

In [ ]:
# Importação de bibliotecas fundamentais
import os
import time
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Pré-processamento e Métricas
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, make_scorer

# Modelos de Classificação (Requisitos do Enunciado)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

# Tentar importar XGBoost (Opcional, mas recomendado)
try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("XGBoost não detetado. Será ignorado na comparação.")

# Configuração de caminhos
DATA_PATH = 'hand_landmarks_dataset.csv'
MODELS_DIR = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)

## 1. Carregamento e Análise Exploratória dos Dados

Nesta etapa, carregamos o ficheiro CSV contendo os *landmarks*. É crucial verificar o equilíbrio das classes, pois um dataset desequilibrado pode enviesar o modelo a prever apenas as classes maioritárias.

Removemos colunas auxiliares que não são características geométricas (como `hand` ou `label` original) para criar a matriz de características $X$.

In [ ]:
# Carregar Dataset
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"O ficheiro {DATA_PATH} não foi encontrado. Execute a extração de landmarks primeiro.")

df = pd.read_csv(DATA_PATH)

print(f"Dimensões do Dataset: {df.shape}")
print("\nPrimeiras 5 linhas:")
display(df.head())

# Verificar equilíbrio das classes
plt.figure(figsize=(12, 5))
sns.countplot(x='label', data=df, order=sorted(df['label'].unique()))
plt.title('Distribuição de Amostras por Letra (Classe)')
plt.xlabel('Letra')
plt.ylabel('Contagem')
plt.show()

## 2. Pré-processamento de Dados

Antes do treino, realizamos três passos fundamentais:
1.  **Label Encoding:** Converter as etiquetas de texto (ex: 'A', 'B') para números inteiros (0, 1), formato necessário para a maioria dos algoritmos.
2.  **Divisão Treino/Teste:** Separamos 20% dos dados para o teste final (hold-out). Utilizamos a opção `stratify=y` para garantir que a proporção de cada letra se mantém igual no treino e no teste.
3.  **Feature Scaling (Normalização):** Aplicamos `StandardScaler` para colocar todas as coordenadas numa escala comum (média 0, desvio padrão 1). Isto é obrigatório para modelos baseados em distância (KNN, SVM) e gradiente (Redes Neuronais).

In [ ]:
# Separar Features (X) e Target (y)
# Removemos 'label' (target) e 'hand' (não discriminativo se usarmos apenas a geometria relativa)
X = df.drop(columns=['label', 'hand'], errors='ignore').values
y_labels = df['label'].values

# 1. Label Encoding
le = LabelEncoder()
y = le.fit_transform(y_labels)
print(f"Classes identificadas: {len(le.classes_)}")
print(f"Mapeamento: {list(zip(range(5), le.classes_[:5]))} ...")

# 2. Divisão Treino / Teste (80% Treino, 20% Teste)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Normalização (Scaling)
scaler = StandardScaler()
# Ajustamos o scaler APENAS ao conjunto de treino para evitar data leakage
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Dataset de Treino: {X_train_s.shape}")
print(f"Dataset de Teste: {X_test_s.shape}")

## 3. Comparação de Modelos (Cross-Validation)

Testaremos os algoritmos exigidos no enunciado:
* **Random Forest:** Robusto e lida bem com não-linearidade.
* **SVM (Support Vector Machines):** Eficaz em espaços de alta dimensão.
* **KNN (K-Nearest Neighbors):** Simples, baseado em similaridade.
* **Decision Tree:** Modelo base simples e interpretável.
* **Neural Network (MLP):** Capaz de aprender padrões complexos.
* *XGBoost (se disponível):* Variante otimizada de Boosting.

**Metodologia de Avaliação:**
Utilizamos **Stratified K-Fold Cross Validation** (5 folds). Isto garante que cada modelo é testado em 5 subconjuntos diferentes dos dados de treino, gerando uma estimativa de desempenho robusta (média e desvio padrão).

Também medimos o **tempo de inferência**, pois o enunciado define-o como critério de desempate.

In [ ]:
# Definição dos Modelos Base
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'Neural Network': MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42)
}

if XGB_AVAILABLE:
    models['XGBoost'] = XGBClassifier(eval_metric='mlogloss', use_label_encoder=False, random_state=42)

# Estrutura para guardar resultados
results_data = []

print("A iniciar comparação de modelos (Cross-Validation 5-Folds)...")
print("Isto pode demorar alguns minutos dependendo do CPU.")

for name, model in models.items():
    start_time = time.time()
    
    # Definir métricas de avaliação
    scoring = {'accuracy': 'accuracy', 'f1_weighted': 'f1_weighted'}
    
    # Executar Cross-Validation
    cv_results = cross_validate(model, X_train_s, y_train, cv=5, scoring=scoring, n_jobs=-1)
    
    # Calcular médias
    mean_acc = cv_results['test_accuracy'].mean()
    mean_f1 = cv_results['test_f1_weighted'].mean()
    std_acc = cv_results['test_accuracy'].std()
    
    # Medir tempo de inferência (simulação de 1 amostra em loop para média)
    # Treinamos uma instância rápida para medir tempo de predição
    model.fit(X_train_s[:1000], y_train[:1000]) # fit parcial apenas para teste de tempo
    
    t0 = time.process_time()
    _ = model.predict(X_test_s[:100]) # Prever 100 amostras
    t1 = time.process_time()
    inference_time_ms = ((t1 - t0) / 100) * 1000 # ms por amostra
    
    results_data.append({
        'Model': name,
        'Accuracy Mean': mean_acc,
        'Accuracy Std': std_acc,
        'F1-Score (Weighted)': mean_f1,
        'Inference Time (ms)': inference_time_ms
    })
    
    print(f"-> {name}: Acc={mean_acc:.4f}, F1={mean_f1:.4f}, Inf.Time={inference_time_ms:.4f}ms")

# Criar DataFrame de comparação
df_results = pd.DataFrame(results_data).sort_values(by='Accuracy Mean', ascending=False)
display(df_results)

## 4. Otimização de Hiperparâmetros (Grid Search)

Com base na tabela acima, selecionamos os dois melhores modelos (geralmente Random Forest e SVM ou Neural Network) para afinação fina.

Utilizamos `GridSearchCV` para testar exaustivamente combinações de parâmetros. 
**Nota:** As grelhas foram definidas para serem eficientes em CPU, focando-se nos parâmetros de maior impacto.

In [ ]:
# Seleção dos modelos para Grid Search (Top 2 baseados em accuracy)
top_models = df_results['Model'].head(2).values
print(f"Modelos selecionados para otimização: {top_models}")

best_estimators = {}

# Definição das grelhas de parâmetros (Ajustadas para tempo razoável de treino)
param_grids = {
    'Random Forest': {
        'n_estimators': [100, 200],
        'max_depth': [None, 20],
        'min_samples_split': [2, 5]
    },
    'SVM': {
        'C': [0.1, 1, 10],
        'kernel': ['rbf', 'linear'],
        'gamma': ['scale', 'auto']
    },
    'XGBoost': {
        'n_estimators': [100, 200],
        'learning_rate': [0.01, 0.1],
        'max_depth': [3, 6]
    },
    'Neural Network': {
        'hidden_layer_sizes': [(128,), (128, 64)],
        'activation': ['relu', 'tanh'],
        'alpha': [0.0001, 0.001]
    },
    'KNN': {
        'n_neighbors': [3, 5, 7],
        'weights': ['uniform', 'distance']
    }
}

for model_name in top_models:
    if model_name not in param_grids: 
        continue
        
    print(f"\nOtimizando {model_name}...")
    base_model = models[model_name]
    
    # Grid Search com Cross Validation (cv=3 para ser mais rápido)
    grid = GridSearchCV(base_model, param_grids[model_name], cv=3, scoring='accuracy', n_jobs=-1, verbose=1)
    grid.fit(X_train_s, y_train)
    
    print(f"Melhores parâmetros para {model_name}: {grid.best_params_}")
    print(f"Melhor score (CV): {grid.best_score_:.4f}")
    best_estimators[model_name] = grid.best_estimator_

## 5. Seleção e Avaliação Final do Modelo

Selecionamos o modelo vencedor com base na performance pós-otimização. 
Agora, fazemos o treino final utilizando **todos** os dados de treino disponíveis e avaliamos no conjunto de teste reservado (que o modelo nunca viu).

Apresentamos:
1.  **Classification Report:** Accuracy, Precision, Recall e F1-Score por classe.
2.  **Matriz de Confusão:** Para visualizar onde o modelo comete erros.

In [ ]:
# Escolha automática do melhor modelo entre os otimizados
final_model_name = None
final_model = None
best_final_score = -1

print("Resultados após GridSearch:")
for name, model in best_estimators.items():
    # Avaliar rapidamente no set de validação/teste para decisão final
    acc = model.score(X_test_s, y_test)
    print(f"{name}: Accuracy Teste = {acc:.4f}")
    if acc > best_final_score:
        best_final_score = acc
        final_model = model
        final_model_name = name

print(f"\n🏆 Modelo Vencedor Selecionado: {final_model_name} (Acc: {best_final_score:.4f})")

# Avaliação Detalhada
y_pred = final_model.predict(X_test_s)
print("\nClassification Report Final:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Matriz de Confusão Visual
plt.figure(figsize=(15, 12))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f'Matriz de Confusão - {final_model_name}')
plt.xlabel('Previsto')
plt.ylabel('Verdadeiro')
plt.show()

## 6. Persistência dos Artefactos (Fase 3 Prep)

Para a Fase 3 (API Flask) e Fase 4 (Aplicação Cliente), precisamos de exportar:
1.  **O Modelo:** O classificador treinado.
2.  **O Scaler:** Para normalizar os novos dados exatamente como no treino.
3.  **O LabelEncoder:** Para converter as predições numéricas (0, 1...) de volta para letras ('A', 'B'...). 

Estes ficheiros serão guardados na pasta `models/`.

In [ ]:
# Guardar os artefactos
joblib.dump(final_model, os.path.join(MODELS_DIR, 'best_model.pkl'))
joblib.dump(scaler, os.path.join(MODELS_DIR, 'scaler_hand_sign.pkl'))
joblib.dump(le, os.path.join(MODELS_DIR, 'label_encoder.pkl'))

print("✅ Artefactos guardados com sucesso:")
print(f"1. Modelo: {os.path.join(MODELS_DIR, 'best_model.pkl')}")
print(f"2. Scaler: {os.path.join(MODELS_DIR, 'scaler_hand_sign.pkl')}")
print(f"3. Encoder: {os.path.join(MODELS_DIR, 'label_encoder.pkl')}")

print("\nPróximo Passo: Utilizar estes ficheiros na criação da API Flask (app.py).")